# Agent Memory, Zero to Hero — Oracle AI Database Edition

## A real Oracle-backed memory core with the current MemoRizz SDK

This companion to `agent_memory.ipynb` keeps the same memory-first
architecture but replaces the isolated filesystem store with a running
Oracle AI Database. It exercises the package-owned runtime bootstrap,
provider preflight, vector-dimension validation, every major memory
family, observability, and transactional scoped cleanup.

It is intentionally a separate notebook: Oracle readiness failures are
surfaced as failures. The notebook never silently switches providers.

This notebook is standalone enough to run first, although the
filesystem edition gives a slower conceptual introduction. Here the
emphasis is the boundary between memory semantics and production
persistence: database readiness, embedding/schema compatibility,
transactional links, tenant-scoped operations, observability, and
cleanup.

**What you will build and verify**

- a persisted `MEMAGENT` reconstructed in another Python object;
- scoped conversation, persona, entity, and knowledge records;
- vector retrieval through Oracle rather than an in-process index;
- durable workflows and reviewed skills;
- governed semantic-cache reuse and source-linked compaction;
- typed shared-memory coordination; and
- capability evidence followed by exact transactional cleanup.

This is an integration tutorial, not a paper benchmark. Passing it
proves the configured components work together; it does not establish
retrieval accuracy on your production corpus.

## Why Oracle for an agent memory core?

```mermaid
flowchart LR
  A[MemAgent] --> P[OracleProvider]
  P --> R[(Relational memory units)]
  P --> V[(VECTOR columns and search)]
  P --> T[Transactional summary links and cleanup]
  R --> C[Conversation, entities, workflows, skills, audit]
  V --> K[Knowledge, semantic cache, retrieval]
  T --> G[Governance and lifecycle]
```

Oracle is useful when one system must combine transactional metadata,
tenant-scoped records, vector retrieval, audit history, and lifecycle
operations. It does not remove the need to evaluate retrieval quality
or design memory policy.

| Concern | Filesystem tutorial | Oracle edition |
|---|---|---|
| Setup | Temporary directory | Docker/runtime plus schema credentials |
| Vector execution | Portable exact path in process | Oracle VECTOR search and optional indexes |
| Transactions | File-level operations | Atomic database operations for linked state |
| Scaling/operations | Best for local or single-node use | Connection pool, privileges, PDB, indexes, vector memory |
| Failure policy | Local I/O errors | Fail-closed preflight before memory writes |

A database does not decide what deserves to become memory. MemoRizz
still owns formation, scope, retrieval policy, cache admission,
compaction, learning, and forgetting; Oracle provides a durable engine
capable of enforcing and querying the resulting records.

## Prerequisites and cost boundary

Install and prepare the local runtime once:

```bash
python -m pip install -e ".[oracle]"
memorizz oracle install
memorizz oracle setup
```

Required environment variables:

- `ORACLE_USER`, `ORACLE_PASSWORD`, `ORACLE_DSN`
- `OPENAI_API_KEY` for the default 384-dimensional OpenAI embedding lane

The reasoning model remains deterministic and local unless
`MEMORIZZ_TUTORIAL_LLM=openai` is set. To use Oracle's configured ONNX
model instead of external embeddings, set
`MEMORIZZ_ORACLE_IN_DATABASE_EMBEDDING=1`.

No credentials are printed or serialized into the agent definition.

The default lane uses the API only for embeddings; reasoning stays
local and deterministic. Set `MEMORIZZ_TUTORIAL_LLM=openai` only when
you want variable hosted-model responses and accept the additional
generation cost. Set `MEMORIZZ_TUTORIAL_KEEP_DATA=1` only when you
intentionally want tutorial rows to survive cleanup.

**Vector dimensions are a schema contract.** Every persisted Oracle
VECTOR column must match the configured embedder. This notebook defaults
to the package runtime's 384-dimensional schema and lets another
installation override it with
`MEMORIZZ_TUTORIAL_EMBEDDING_DIMENSIONS`. Never truncate or pad vectors
merely to make an insert succeed; migrate or configure consistently.

In [1]:
import os
import uuid
from importlib.metadata import version

import memorizz
from dotenv import find_dotenv, load_dotenv
from memorizz import (
    ContextPolicy,
    EntityMemory,
    KnowledgeBase,
    LocalOracleRuntime,
    MemAgent,
    MemAgentBuilder,
    MemoryType,
    OracleProvider,
    Persona,
    RoleType,
    SharedMemory,
    Toolbox,
    governed_tool,
)
from memorizz.long_term.procedural.skillbox import SkillStatus
from memorizz.long_term.procedural.workflow import Workflow


env_file = find_dotenv(usecwd=True)
if env_file:
    load_dotenv(env_file, override=False)


def env_flag(name: str, default: bool = False) -> bool:
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "on"}


USE_OPENAI_LLM = os.getenv("MEMORIZZ_TUTORIAL_LLM", "local").lower() == "openai"
USE_IN_DATABASE_EMBEDDING = env_flag("MEMORIZZ_ORACLE_IN_DATABASE_EMBEDDING")
KEEP_DATA = env_flag("MEMORIZZ_TUTORIAL_KEEP_DATA")
# The package-owned local Oracle schema is created at 384 dimensions.
# Override this tutorial-specific value when your schema was provisioned
# differently; preflight will reject any mismatch before writes.
EMBEDDING_DIMENSIONS = int(os.getenv("MEMORIZZ_TUTORIAL_EMBEDDING_DIMENSIONS", "384"))
OPENAI_MODEL = os.getenv("MEMORIZZ_TUTORIAL_MODEL", "gpt-5-mini")

required = ["ORACLE_USER", "ORACLE_PASSWORD", "ORACLE_DSN"]
if not USE_IN_DATABASE_EMBEDDING:
    required.append("OPENAI_API_KEY")
if USE_OPENAI_LLM:
    required.append("OPENAI_API_KEY")
missing = sorted({name for name in required if not os.getenv(name)})
if missing:
    raise RuntimeError(f"Missing required environment variables: {missing}")

RUN_ID = os.getenv("MEMORIZZ_TUTORIAL_RUN_ID", uuid.uuid4().hex[:10])
MEMORY_ID = f"oracle-zero-to-hero-{RUN_ID}"
USER_ID = f"oracle-guide-user-{RUN_ID}"
THREAD_ID = "engineering-copilot"

print(
    {
        "memorizz_version": version("memorizz"),
        "package": "memorizz",
        "reasoning_lane": "openai" if USE_OPENAI_LLM else "deterministic-local",
        "embedding_lane": (
            "oracle-onnx"
            if USE_IN_DATABASE_EMBEDDING
            else f"openai-{EMBEDDING_DIMENSIONS}"
        ),
        "environment_file_found": bool(env_file),
        "run_id": RUN_ID,
        "keep_data": KEEP_DATA,
    }
)

{'memorizz_version': '0.6.0', 'package': 'memorizz', 'reasoning_lane': 'deterministic-local', 'embedding_lane': 'openai-384', 'environment_file_found': True, 'run_id': 'be7b460f36', 'keep_data': False}


## 1 · Start or verify the local Oracle runtime

`ensure_ready()` is idempotent. It starts an existing stopped
container and waits for database readiness. Because
`provision_if_missing=False`, an absent container produces an
actionable error instead of unexpectedly downloading a large image.

**Read the output:** `ok=True`, `state=running`, and an action of either
`none` or `started` prove that the named container is healthy. This is
runtime readiness only; database privileges and vector compatibility
are checked next.

**Common failures:** Docker daemon unavailable, a differently named
container, a port collision, or an unready listener. Resolve these
explicitly. A tutorial that silently falls back to filesystem could
appear green while never testing Oracle.

In [2]:
runtime = LocalOracleRuntime.from_env(provision_if_missing=False)
runtime_report = runtime.ensure_ready()
print(
    {
        "ok": runtime_report["ok"],
        "container": runtime_report["container"],
        "state": runtime_report["state"],
        "action": runtime_report["action"],
    }
)

{'ok': True, 'container': 'erpa-memorizz-oracle', 'state': 'running', 'action': 'none'}


## 2 · Construct and preflight the Oracle memory provider

`index_policy="lazy"` avoids eagerly attempting every vector index.
Exact vector search remains available while indexes are absent. The
preflight report checks database/PDB state, privileges, embedding
model and dimensions, vector columns, vector memory, and index status.

| Preflight signal | Why it matters |
|---|---|
| Product and `version_full` | Reproduces database behavior precisely |
| PDB/service state | Confirms the intended pluggable database is open |
| Privileges/schema | Prevents failures after partial ingestion |
| Embedder and VECTOR dimensions | Prevents invalid or incomparable vectors |
| `VECTOR_MEMORY_SIZE` and indexes | Explains indexed versus exact-search behavior |
| Diagnostics | Produces one actionable failure report |

**Read the output:** diagnostics must be empty and `ok` true. The
explicit dimension validator is intentionally redundant with preflight
because writes made with the wrong embedding shape would corrupt the
experiment. `exact_search_fallback=True` means correctness can continue
without an HNSW index, although latency still needs measurement.

In [3]:
provider_options = {
    "index_policy": "lazy",
    "in_database_embedding": USE_IN_DATABASE_EMBEDDING,
}
if not USE_IN_DATABASE_EMBEDDING:
    provider_options.update(
        {
            "embedding_provider": "openai",
            "embedding_config": {
                "model": "text-embedding-3-small",
                "dimensions": EMBEDDING_DIMENSIONS,
            },
        }
    )

provider = OracleProvider.from_env(**provider_options)
preflight = provider.preflight()
print(
    {
        key: preflight.get(key)
        for key in (
            "ok",
            "database_product",
            "version_full",
            "pdb_open_mode",
            "embedding_model",
            "embedding_dimensions",
            "vector_memory_size",
            "index_policy",
            "exact_search_fallback",
            "diagnostics",
        )
    }
)
if not preflight.get("ok"):
    raise RuntimeError("Oracle preflight failed; inspect the diagnostics above.")
if not USE_IN_DATABASE_EMBEDDING:
    provider.validate_vector_schema_dimensions(EMBEDDING_DIMENSIONS)

{'ok': True, 'database_product': 'Oracle AI Database 26ai Free', 'version_full': '23.26.0.0.0', 'pdb_open_mode': None, 'embedding_model': None, 'embedding_dimensions': None, 'vector_memory_size': '268435456', 'index_policy': 'lazy', 'exact_search_fallback': True, 'diagnostics': []}


## 3 · A local teaching model, with an optional hosted lane

Oracle is the real persistence and retrieval backend in both lanes.
The deterministic model keeps the tutorial cheap and makes lifecycle
assertions reproducible; it is not presented as a model-quality test.

This separation is important for applied research. The storage and
retrieval lane can be tested while holding the reader constant. A later
model A/B run can reuse the same Oracle snapshot rather than paying to
ingest and embed it again. Record both lanes separately when measuring
cost and latency.

In [4]:
class TutorialModel:
    model = "oracle-tutorial-model"

    def __init__(self):
        self.calls = 0
        self._usage = {}

    def generate(self, messages, tools=None):
        self.calls += 1
        text = "\n".join(
            str(message.get("content") or "")
            for message in messages
            if isinstance(message, dict)
        )
        lowered = text.lower()
        self._usage = {
            "prompt_tokens": max(1, len(text) // 4),
            "completion_tokens": 16,
            "total_tokens": max(1, len(text) // 4) + 16,
        }
        if "comprehensive but concise summary" in lowered:
            return "Ada owns retrieval-api and is migrating RAG to Oracle AI Database."
        users = [
            str(message.get("content") or "")
            for message in messages
            if isinstance(message, dict) and message.get("role") == "user"
        ]
        latest = users[-1].lower() if users else ""
        if "what project" in latest or "who am i" in latest:
            if "ada" in lowered and "oracle" in lowered:
                return "You are Ada, migrating the RAG stack to Oracle AI Database."
            return "I do not have that information in this request."
        return "Recorded in the Oracle-backed memory scope."

    def get_config(self):
        return {"provider": "tutorial", "model": self.model}

    def get_last_usage(self):
        return dict(self._usage)


def make_model():
    if not USE_OPENAI_LLM:
        return TutorialModel()
    from memorizz.llms.openai import OpenAI

    return OpenAI(model=OPENAI_MODEL, reasoning_effort="none")

---
# Part I · Persist an agent and an episodic thread

Oracle stores both the `MEMAGENT` definition and the scoped
`CONVERSATION_MEMORY` rows. A restored agent receives a model override
because credentials and executable provider objects are not persisted.

The host supplies `memory_id`, `user_id`, and `thread_id` on every turn.
Oracle filters scope before retrieval; the model is never trusted to
choose its tenant. `build_and_save()` writes the serializable agent
configuration, while `MemAgent.load(...)` reconstructs it with a
trusted runtime model client.

**Read the output:** the second turn recalls Ada, the reconstructed
agent recalls the project again, and every history row satisfies the
user/thread assertions. Those checks demonstrate durable episodic
continuity and isolation, not merely a plausible generated sentence.

In production, test a negative scope too: another user and thread must
retrieve zero of these rows.

In [5]:
agent = (
    MemAgentBuilder()
    .with_name(f"Oracle Memo {RUN_ID}")
    .with_instruction("Be concise and ground durable claims in retrieved memory.")
    .with_model(make_model())
    .with_memory_provider(provider)
    .with_memory_ids(MEMORY_ID)
    .with_entity_memory(True)
    .with_context_policy(ContextPolicy(progressive_tool_disclosure=True, tool_top_k=3))
    .with_automations_enabled(False)
    .build_and_save()
)
scope = {"memory_id": MEMORY_ID, "user_id": USER_ID, "thread_id": THREAD_ID}
print(agent.run("I'm Ada. I am migrating our RAG stack to Oracle AI Database.", **scope))
print(agent.run("What project am I working on and who am I?", **scope))

restored = MemAgent.load(agent.agent_id, memory_provider=provider, model=make_model())
print(restored.run("What project were we discussing?", **scope))

Recorded in the Oracle-backed memory scope.


You are Ada, migrating the RAG stack to Oracle AI Database.


You are Ada, migrating the RAG stack to Oracle AI Database.


In [6]:
history = provider.retrieve_conversation_history_ordered_by_timestamp(
    memory_id=MEMORY_ID,
    memory_type=MemoryType.CONVERSATION_MEMORY,
    user_id=USER_ID,
    thread_id=THREAD_ID,
)
print([(row.get("role"), row.get("content")) for row in history])
assert all(row.get("user_id") == USER_ID for row in history)
assert all((row.get("thread_id") or row.get("conversation_id")) == THREAD_ID for row in history)

[('user', "I'm Ada. I am migrating our RAG stack to Oracle AI Database."), ('assistant', 'Recorded in the Oracle-backed memory scope.'), ('user', 'What project am I working on and who am I?'), ('assistant', 'You are Ada, migrating the RAG stack to Oracle AI Database.'), ('user', 'What project were we discussing?'), ('assistant', 'You are Ada, migrating the RAG stack to Oracle AI Database.')]


---
# Part II · Semantic memory in Oracle

Persona and entity records retain version, confidence, source, and
tenant metadata alongside their vectors. Knowledge-base chunks use the
same provider and can be attached to the persisted agent.

These semantic representations answer different questions:

- **Persona:** who the agent is; updates are versioned and authorized.
- **Entity:** structured current claims about a service or person;
  attributes retain source and confidence.
- **Knowledge base:** what an approved source passage says; chunks
  retain namespace and document lineage.

The entity output should show Ada as owner and Oracle as the database.
The retrieval output should rank the post-rebuild verification passage.
A score proves ranking behavior, while the text and source metadata are
what make a later answer groundable.

**Lifecycle:** supersede stale entity claims, version source documents,
and retain persona evolution triggers. Oracle durability does not make
old information current automatically.

In [7]:
persona = Persona(
    name="Oracle Memo",
    role=RoleType.TECHNICAL_EXPERT,
    goals="Help engineers build grounded, observable memory systems.",
    background="An AI platform engineer specializing in Oracle vector search.",
)
agent.set_persona(persona)
persona.update(
    updates={"goals": "Help engineers build grounded, observable, and cost-aware memory systems."},
    change_trigger={
        "reason": "Cost governance was added to the platform requirements.",
        "source_type": "user_feedback",
        "source_id": f"decision-{RUN_ID}",
        "agent_id": agent.agent_id,
    },
    provider=provider,
)

entities = EntityMemory(provider)
entities.upsert_entity(
    entity_id=f"retrieval-api-{RUN_ID}",
    name="retrieval-api",
    entity_type="service",
    attributes=[
        {"name": "owner", "value": "Ada", "confidence": 0.98, "source": "service-catalog"},
        {"name": "database", "value": "Oracle AI Database", "confidence": 0.98, "source": "architecture-record"},
    ],
    memory_id=MEMORY_ID,
    user_id=USER_ID,
)
print(entities.list_entities(memory_id=MEMORY_ID, user_id=USER_ID))

[{'_id': 'e1d42e27-4a5b-430f-9806-de57781f973b', 'entity_id': 'retrieval-api-be7b460f36', 'name': 'retrieval-api', 'entity_type': 'service', 'attributes': [{'name': 'owner', 'value': 'Ada', 'confidence': 0.98, 'source': 'service-catalog', 'created_at': '2026-08-23T21:19:44.405546', 'updated_at': '2026-08-23T21:19:44.405546'}, {'name': 'database', 'value': 'Oracle AI Database', 'confidence': 0.98, 'source': 'architecture-record', 'created_at': '2026-08-23T21:19:44.405546', 'updated_at': '2026-08-23T21:19:44.405546'}], 'relations': [], 'metadata': {}, 'memory_id': 'oracle-zero-to-hero-be7b460f36', 'agent_id': None, 'user_id': 'oracle-guide-user-be7b460f36', 'created_at': '2026-08-23T22:19:44.602401', 'updated_at': '2026-08-23T22:19:44.602401'}]


In [8]:
kb = KnowledgeBase(provider)
kb_id = kb.ingest_knowledge(
    (
        "Before rebuilding a production vector index, capture a schema snapshot and "
        "verify capacity. Preserve a tested rollback path. After rebuilding, validate "
        "both exact and indexed vector search before closing the maintenance window."
    ),
    namespace=f"oracle-runbook-{RUN_ID}",
    chunking_strategy="sentence",
    chunk_size=180,
    user_id=USER_ID,
)
kb.attach_to_agent(agent, kb_id)
hits = provider.retrieve_by_query(
    "What must be validated after rebuilding a vector index?",
    memory_store_type=MemoryType.KNOWLEDGE_BASE,
    namespace=f"oracle-runbook-{RUN_ID}",
    user_id=USER_ID,
    limit=3,
)
print([(round(row.get("score", 0.0), 3), row["content"]) for row in hits])
assert hits

[(0.775, 'After rebuilding, validate both exact and indexed vector search before closing the maintenance window.'), (0.68, 'Before rebuilding a production vector index, capture a schema snapshot and verify capacity. Preserve a tested rollback path.')]


---
# Part III · Procedural memory in Oracle

Deterministic tool registration creates a complete strict schema
without constructing another LLM. Workflow storage computes a
canonical trajectory identity. Authored skills use first-class
Skillbox persistence without enabling continual learning.

The three units have different authority. `TOOLBOX` declares what can
execute; `WORKFLOW_MEMORY` records or describes an ordered trajectory;
`SKILLBOX` supplies reviewed instruction for a class of tasks. A
persisted schema never recreates executable Python—the trusted host
must bind the callable after restart.

**Read the output:** the zero-argument health schema has
`additionalProperties: false`; the workflow has a durable row and
canonical hash; and scoped skill retrieval returns the reviewed Oracle
verification playbook.

Continual learning is deliberately disabled. In a learning deployment,
successful trajectories may propose candidate skills, but instruction
hierarchy makes automatic promotion risky. Require verified outcomes,
minimum support, shadow evaluation, versioning, and reversible demotion.

In [9]:
@governed_tool(deterministic=True, side_effects=False, domains=("oracle-health",))
def oracle_health() -> dict:
    '''Return a read-only status derived from the completed provider preflight.'''
    return {
        "ok": bool(preflight.get("ok")),
        "version_full": preflight.get("version_full"),
        "exact_search_fallback": preflight.get("exact_search_fallback"),
    }


toolbox = Toolbox.from_functions(
    [oracle_health],
    memory_provider=provider,
    agent_id=agent.agent_id,
    user_id=USER_ID,
    augment=False,
)
print(toolbox.get_tool_by_name("oracle_health")["input_schema"])

workflow = Workflow(
    name="oracle-vector-index-verification",
    description="Verify Oracle readiness before and after an index change.",
    memory_id=MEMORY_ID,
    agent_id=agent.agent_id,
    user_id=USER_ID,
    user_query="Verify the Oracle vector-index lifecycle.",
)
workflow.add_step("preflight", {"tool": "oracle_health", "arguments": {}})
workflow.add_step("exact_search", {"tool": "verify_vector_search", "arguments": {"mode": "exact"}})
workflow.add_step("indexed_search", {"tool": "verify_vector_search", "arguments": {"mode": "indexed"}})
workflow_row_id = workflow.store_workflow(provider)

skill = {
    "name": f"oracle/vector-index-verification-{RUN_ID}",
    "description": "Verify Oracle vector search around an index lifecycle change.",
    "content": "Run preflight, verify exact search, verify indexed search, then compare evidence.",
    "preconditions": ["Oracle preflight is successful"],
    "tools_used": ["oracle_health"],
    "queries": ["verify Oracle vector search", "check vector index readiness"],
    "user_id": USER_ID,
    "status": SkillStatus.ACTIVE.value,
}
skilled_agent = (
    MemAgentBuilder()
    .with_name(f"Oracle Skilled Memo {RUN_ID}")
    .with_instruction("Use a relevant reviewed skill for Oracle procedures.")
    .with_model(make_model())
    .with_memory_provider(provider)
    .with_memory_ids(MEMORY_ID)
    .with_skills([skill])
    .with_skill_retrieval(enabled=True, top_k=2, min_similarity=0.0)
    .with_continual_learning(enabled=False)
    .with_automations_enabled(False)
    .build_and_save()
)
skill_hits = skilled_agent.skillbox.retrieve_skills_by_query(
    "How do I verify Oracle vector search?",
    limit=2,
    min_similarity=0.0,
    user_id=USER_ID,
)
print(
    {
        "workflow_row_id": workflow_row_id,
        "canonical_hash": workflow.canonical_hash,
        "skills": [hit.skill.name for hit in skill_hits],
    }
)

{'type': 'object', 'properties': {}, 'additionalProperties': False, 'required': []}


{'workflow_row_id': '293bc570-87da-4257-b718-561a487153e7', 'canonical_hash': '6ee2891c4c376e04744631766a5fb86f45debc2e2f08d80377463fe775026551', 'skills': ['oracle/vector-index-verification-be7b460f36']}


---
# Part IV · Cache, compaction, and shared memory

These operations are where Oracle's unified relational/vector store is
especially useful: cache metadata, summary/source links, and
coordination state share a transactional provider boundary.

**Semantic cache.** The first deterministic read misses and writes; the
second hits. Inspection exposes the matched key, fingerprints,
similarity, age, TTL, and invalidation domains. Cache reuse is scoped
by user/session and data version; similarity alone never establishes
freshness.

**Compaction.** `generate_summaries` writes one scoped summary and marks
its source messages atomically. The output includes source IDs, period
bounds, and count so the summary remains expandable and auditable.
Compression reduces repeated prompt tokens; it does not delete history.

**Coordination.** Shared memory records a typed command and report with
participant, workflow, tenant, trace, and citation identity. It is a
workflow blackboard—not global application state.

These mechanisms also illustrate forgetting: cache entries expire or
invalidate, summaries replace default replay of old turns, skills can
be demoted, claims can be superseded, and scoped retention can delete
rows. Each type needs its own forgetting policy.

In [10]:
cache_model = TutorialModel() if not USE_OPENAI_LLM else make_model()
cache_agent = (
    MemAgentBuilder()
    .with_name(f"Oracle Cache Memo {RUN_ID}")
    .with_instruction("Answer deterministic read-only database questions.")
    .with_model(cache_model)
    .with_memory_provider(provider)
    .with_memory_ids(MEMORY_ID)
    .with_semantic_cache(enabled=True, threshold=0.95, scope="session")
    .with_automations_enabled(False)
    .build_and_save()
)
cache_context = {"cache_domains": ["oracle-runbook"], "data_version": RUN_ID}
cache_scope = {**scope, "context": cache_context}
cache_query = "What project am I working on and who am I?"
print(cache_agent.run(cache_query, **cache_scope))
print(cache_agent.run(cache_query, **cache_scope))
inspection = cache_agent.inspect_semantic_cache(
    cache_query,
    user_id=USER_ID,
    thread_id=THREAD_ID,
    context=cache_context,
)
print({"stats": cache_agent.semantic_cache_stats(), "inspection": inspection.to_dict()})
if not USE_OPENAI_LLM:
    assert cache_model.calls == 1 and inspection.hit

You are Ada, migrating the RAG stack to Oracle AI Database.
You are Ada, migrating the RAG stack to Oracle AI Database.
{'stats': {'enabled': True, 'hits': 1, 'misses': 1, 'bypasses': 0, 'writes': 1, 'evictions': 0, 'size': 1, 'bypass_reasons': {}, 'last_hit': {'cache_key': '2c5ed234-b32e-5fe5-baa5-e3bf20818f98', 'query': 'What project am I working on and who am I?', 'similarity': 1.0, 'age_seconds': 0.017261743545532227, 'agent_id': 'c0979f1a-515e-5168-8e2a-55df89e5f4b8', 'memory_id': 'oracle-zero-to-hero-be7b460f36', 'session_id': 'engineering-copilot', 'user_id': 'oracle-guide-user-be7b460f36', 'metadata': {'fingerprints': {'model': '3a9d6a78fae9189dea4e6780effc92ee76f33b0d78ff1b0a912b3ea53f25731f', 'prompt': '8e1404311d0337b842afd636bb9266bfdf90880dd0a09752d569921e14e3d599', 'tool_schema': 'c8177d5db0a1484f7e2868070e486b7d1c0c4f0ab5d86f7263c8bb4540bc13eb', 'completion_policy': 'e0b81db2190bb1955538db6433846a0880923ce028738c1951fd21fc4edad1b0', 'data_version': 'be7b460f36', 'request

In [11]:
summary_ids = agent.generate_summaries(
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id=THREAD_ID,
    days_back=7,
    max_memories_per_summary=20,
)
summary = agent.fetch_context_summary(
    summary_ids[0],
    memory_id=MEMORY_ID,
    user_id=USER_ID,
    thread_id=THREAD_ID,
)
print(
    {
        "summary_id": summary_ids[0],
        "source_message_ids": summary["source_message_ids"],
        "memory_units_count": summary["memory_units_count"],
        "period_start": summary["period_start"],
        "period_end": summary["period_end"],
    }
)

{'summary_id': '4e41c409-f571-4a82-922f-6d22735e118b', 'source_message_ids': ['1b004c60-324e-48df-8f8f-2e390ebf8bf7', '999675fb-5aa6-437d-b883-d18b5ef04673', '796f6c2f-b8a7-4445-9ab0-cd6878f0e8b2', '5bbea423-3844-4846-9e17-70248661051d', 'ac428148-8355-4dc9-807d-f24432cf7c22', '87e898f4-7170-4e5c-8e70-4b9ca6448a59'], 'memory_units_count': 6, 'period_start': 1787519982.342817, 'period_end': 1787519984.025026}


In [12]:
shared = SharedMemory(provider)
shared_id = shared.create_shared_session(
    root_agent_id=agent.agent_id,
    delegate_agent_ids=[skilled_agent.agent_id],
    workflow_id=f"oracle-memory-review-{RUN_ID}",
    user_id=USER_ID,
    trace_id=f"oracle-trace-{RUN_ID}",
)
shared.post_command(
    shared_id,
    agent_id=agent.agent_id,
    command_id="verify-1",
    target_agent_id=skilled_agent.agent_id,
    instructions="Verify exact and indexed vector search.",
)
shared.post_report(
    shared_id,
    agent_id=skilled_agent.agent_id,
    command_id="verify-1",
    findings="Preflight passed and the provider returned scoped vector evidence.",
    citations=[summary_ids[0]],
)
print(shared.get_blackboard_entries(shared_id))

[{'memory_id': '58a1acb5-cbc1-4837-9c44-c7dd8cf366a0', 'agent_id': 'aba63d6a-3003-5dd5-8864-e2c57104f62a', 'content': {'message_id': 'dd65f054-d10e-4db8-9b5b-b4f3dfc7052c', 'message_type': 'COMMAND', 'created_at': '2026-08-23T21:19:48.965998', 'payload': {'command_id': 'verify-1', 'target_agent_id': 'abdbc26f-5159-5bff-8fa9-3f4f28bfc3dd', 'instructions': 'Verify exact and indexed vector search.', 'priority': 3, 'dependencies': [], 'metadata': {}}}, 'entry_type': 'COMMAND', 'created_at': '2026-08-23T22:19:48.970316'}, {'memory_id': '5d5f5a5f-e212-4985-9e3f-899e05f44a18', 'agent_id': 'abdbc26f-5159-5bff-8fa9-3f4f28bfc3dd', 'content': {'message_id': 'a22ae28f-302e-403e-a71d-08d3dd93006e', 'message_type': 'REPORT', 'created_at': '2026-08-23T21:19:48.974494', 'payload': {'command_id': 'verify-1', 'agent_id': 'abdbc26f-5159-5bff-8fa9-3f4f28bfc3dd', 'findings': 'Preflight passed and the provider returned scoped vector evidence.', 'citations': ['4e41c409-f571-4a82-922f-6d22735e118b'], 'gaps': 

---
# Part V · Operational evidence and cleanup

`observability_summary` is tenant-scoped and content-light. `preflight`
proves database readiness. `delete_scope` removes package-owned rows in
one transaction and reports per-table counts. Knowledge-base chunks and
the standalone shared session are explicitly removed because their
physical IDs are intentionally independent of the conversation scope.

**Read the output:** the capability report must identify
`OracleProvider`, the exact database product/version, lazy indexing,
and the local teaching model. Observability should count the six
conversation rows, one workflow, one summary, and the context budget
without dumping full tenant content.

Cleanup defaults to on because tutorials should be repeatable and
should not leave unexplained database state. The final report lists
counts per store and a total. In a production deletion workflow, retain
the report as audit evidence and make the exact scope visible to the
approving host before executing it.

### Production checklist

- Run preflight at deployment and whenever embedding configuration changes.
- Pin database, embedding model, dimensions, schema migration, and index policy.
- Scope every read before vector top-k selection.
- Measure exact and indexed retrieval latency and recall.
- Monitor connection-pool pressure, VECTOR memory, index state, and fallbacks.
- Keep credentials in process-level secret injection, never agent rows.
- Test summary atomicity, cache invalidation, reload, and scoped deletion.

In [13]:
capability_report = agent.capability_report(preflight=True)
agent_capabilities = capability_report["agent"]
provider_preflight = agent_capabilities.get("provider_preflight") or {}
print(
    {
        "package": capability_report["package"],
        "version": capability_report["version"],
        "agent": {
            key: agent_capabilities.get(key)
            for key in (
                "memory_provider",
                "llm_provider",
                "llm_model",
                "progressive_tool_disclosure",
                "skill_retrieval",
                "continual_learning",
            )
        },
        "oracle": {
            key: provider_preflight.get(key)
            for key in (
                "ok",
                "database_product",
                "version_full",
                "index_policy",
                "exact_search_fallback",
            )
        },
    }
)
print(agent.observability_summary(MEMORY_ID, USER_ID, thread_id=THREAD_ID))

{'package': 'memorizz', 'version': '0.6.0', 'agent': {'memory_provider': 'OracleProvider', 'llm_provider': 'TutorialModel', 'llm_model': 'oracle-tutorial-model', 'progressive_tool_disclosure': True, 'skill_retrieval': False, 'continual_learning': False}, 'oracle': {'ok': True, 'database_product': 'Oracle AI Database 26ai Free', 'version_full': '23.26.0.0.0', 'index_policy': 'lazy', 'exact_search_fallback': True}}
{'agent_id': 'aba63d6a-3003-5dd5-8864-e2c57104f62a', 'memory_id': 'oracle-zero-to-hero-be7b460f36', 'user_id': 'oracle-guide-user-be7b460f36', 'thread_id': 'engineering-copilot', 'conversation': {'row_count': 6, 'message_count': 6, 'role_counts': {'user': 3, 'assistant': 3}, 'thread_count': 1, 'summarized_count': 6, 'trace_bundle_count': 0, 'trace_event_count': 0, 'first_timestamp': 1787519982.342817, 'last_timestamp': 1787519984.025026}, 'tool_logs': {'count': 0, 'failure_count': 0, 'success_count': 0}, 'workflows': {'count': 1, 'outcomes': {'success': 1}}, 'summaries': {'cou

In [14]:
agent_ids = [agent.agent_id, restored.agent_id, skilled_agent.agent_id, cache_agent.agent_id]
for current in (restored, agent, skilled_agent, cache_agent):
    current.close(close_memory_provider=False)

if KEEP_DATA:
    print(
        {
            "kept": True,
            "memory_id": MEMORY_ID,
            "user_id": USER_ID,
            "agent_ids": sorted(set(agent_ids)),
        }
    )
else:
    for chunk in kb.retrieve_knowledge(kb_id):
        provider.delete_by_id(str(chunk.get("_id") or chunk.get("id")), MemoryType.KNOWLEDGE_BASE)
    provider.delete_by_id(shared_id, MemoryType.SHARED_MEMORY)
    cleanup = provider.delete_scope(
        memory_id=MEMORY_ID,
        user_id=USER_ID,
        agent_ids=sorted(set(agent_ids)),
    )
    print("Scoped cleanup:", cleanup)

provider.close()

Scoped cleanup: {'ok': True, 'scope': {'memory_id': 'oracle-zero-to-hero-be7b460f36', 'user_id': 'oracle-guide-user-be7b460f36', 'user_id_supplied': True, 'agent_ids': ['aba63d6a-3003-5dd5-8864-e2c57104f62a', 'abdbc26f-5159-5bff-8fa9-3f4f28bfc3dd', 'c0979f1a-515e-5168-8e2a-55df89e5f4b8']}, 'counts': {'automation_deliveries': 0, 'automation_runs': 0, 'conversation_memory': 6, 'semantic_cache': 1, 'workflow_memory': 1, 'tool_log': 0, 'skillbox': 1, 'summaries': 1, 'entity_memory': 0, 'knowledge_base': 0, 'short_term_memory': 0, 'toolbox': 0, 'personas': 0, 'shared_memory': 0, 'automation_jobs': 0, 'agent_memories': 3, 'agents': 3}, 'total_deleted': 16}


## What this notebook proved

- Oracle runtime readiness and provider preflight are package-owned.
- Conversation and agent definitions survive reconstruction.
- Persona, entity, knowledge, workflow, skill, cache, summary, and
  shared-memory units coexist behind one provider contract.
- Tenant/thread scope is explicit at every user-facing boundary.
- Summary records retain lossless source links.
- Cache reuse is fingerprinted and inspectable.
- Cleanup is exact, transactional, and count-reporting.

This is an integration tutorial, not a retrieval benchmark. Use the
MemoRizz evaluation suite to measure recall, grounding, latency, token
use, and cost on representative workloads.

### Suggested exercises

1. Run once with external embeddings and once with a compatible Oracle
   ONNX embedding model; compare preflight and retrieval timing.
2. Set the tutorial dimension incorrectly and study the fail-closed
   diagnostic, then restore the correct value before any writes.
3. Set `MEMORIZZ_TUTORIAL_KEEP_DATA=1`, inspect scoped counts in the UI
   or CLI, then call `delete_scope` deliberately.
4. Add a second tenant and assert that semantic search cannot retrieve
   the first tenant's entity or conversation rows.
5. Use a representative corpus and report retrieval metrics separately
   from a hosted reader's answer score.